## Setup — SparkSession and data load

In [3]:
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DoubleType, IntegerType

spark = (
    SparkSession.builder.appName("LungCancerAssignment1")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

CSV_PATH = Path.cwd() / "Lung Cancer.csv"
raw_df = (
    spark.read.option("header", True)
    .option("inferSchema", False)
    .csv(str(CSV_PATH))
)
raw_df.printSchema()
print(f"Row count: {raw_df.count():,}")
raw_df.show(3, truncate=False)

root
 |-- id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- diagnosis_date: string (nullable = true)
 |-- cancer_stage: string (nullable = true)
 |-- family_history: string (nullable = true)
 |-- smoking_status: string (nullable = true)
 |-- bmi: string (nullable = true)
 |-- cholesterol_level: string (nullable = true)
 |-- hypertension: string (nullable = true)
 |-- asthma: string (nullable = true)
 |-- cirrhosis: string (nullable = true)
 |-- other_cancer: string (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- end_treatment_date: string (nullable = true)
 |-- survived: string (nullable = true)

Row count: 890,000
+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|id |age |gender|country    |diagnosis_date|cancer_stage|family

## Task 1 — Clean data: drop duplicates, cast types, normalise yes/no fields

In [5]:
YES_NO_COLS = ["family_history"]
INT_COLS = ["id", "cholesterol_level", "hypertension", "asthma", "cirrhosis", "other_cancer", "survived"]
DOUBLE_COLS = ["age", "bmi"]
DATE_COLS = ["diagnosis_date", "end_treatment_date"]


def clean_data(df: DataFrame) -> DataFrame:
    df = df.dropDuplicates()

    for c in INT_COLS:
        df = df.withColumn(c, F.col(c).cast(IntegerType()))
    for c in DOUBLE_COLS:
        df = df.withColumn(c, F.col(c).cast(DoubleType()))
    for c in DATE_COLS:
        df = df.withColumn(c, F.to_date(F.col(c), "yyyy-MM-dd").cast(DateType()))

    for c in YES_NO_COLS:
        df = df.withColumn(
            c,
            F.when(F.lower(F.trim(F.col(c))) == "yes", F.lit(1))
            .when(F.lower(F.trim(F.col(c))) == "no", F.lit(0))
            .otherwise(F.col(c).cast(IntegerType())),
        )
    return df


clean_df = clean_data(raw_df).cache()
clean_df.printSchema()
print(f"Cleaned row count: {clean_df.count():,}")
clean_df.show(3, truncate=False)

root
 |-- id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- diagnosis_date: date (nullable = true)
 |-- cancer_stage: string (nullable = true)
 |-- family_history: integer (nullable = true)
 |-- smoking_status: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- cholesterol_level: integer (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- asthma: integer (nullable = true)
 |-- cirrhosis: integer (nullable = true)
 |-- other_cancer: integer (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- end_treatment_date: date (nullable = true)
 |-- survived: integer (nullable = true)

Cleaned row count: 890,000
+---+----+------+----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|id |age |gender|country   |diagnosis_date|cancer_st

## Task 2 — treatment_duration_days and average per treatment_type

In [7]:
def add_treatment_duration(df: DataFrame) -> DataFrame:
    return df.withColumn(
        "treatment_duration_days",
        F.datediff(F.col("end_treatment_date"), F.col("diagnosis_date")),
    )


def avg_duration_per_treatment(df: DataFrame) -> DataFrame:
    return (
        add_treatment_duration(df)
        .groupBy("treatment_type")
        .agg(F.avg("treatment_duration_days").alias("avg_treatment_duration_days"))
        .orderBy(F.col("avg_treatment_duration_days").desc())
    )


duration_df = add_treatment_duration(clean_df)
duration_df.select("treatment_type", "diagnosis_date", "end_treatment_date", "treatment_duration_days").show(5, truncate=False)
avg_duration_per_treatment(clean_df).show(truncate=False)

+--------------+--------------+------------------+-----------------------+
|treatment_type|diagnosis_date|end_treatment_date|treatment_duration_days|
+--------------+--------------+------------------+-----------------------+
|Combined      |2023-11-29    |2025-01-08        |406                    |
|Radiation     |2023-01-02    |2024-12-27        |725                    |
|Combined      |2017-08-07    |2019-08-03        |726                    |
|Surgery       |2023-03-30    |2024-01-22        |298                    |
|Radiation     |2017-07-29    |2018-08-21        |388                    |
+--------------+--------------+------------------+-----------------------+
only showing top 5 rows
+--------------+---------------------------+
|treatment_type|avg_treatment_duration_days|
+--------------+---------------------------+
|Radiation     |458.40320462900917         |
|Chemotherapy  |458.39540091909953         |
|Combined      |457.8152186120058          |
|Surgery       |457.73744630723

## Task 3 — smoking_status group with the highest survival rate

In [9]:
def smoking_survival_rate(df: DataFrame) -> DataFrame:
    return (
        df.groupBy("smoking_status")
        .agg(
            F.count("*").alias("patients"),
            F.avg("survived").alias("survival_rate"),
        )
        .orderBy(F.col("survival_rate").desc())
    )


def top_smoking_group_by_survival(df: DataFrame) -> str:
    top = smoking_survival_rate(df).first()
    return top["smoking_status"]


smoking_survival_rate(clean_df).show(truncate=False)
print(f"Highest survival rate smoking group: {top_smoking_group_by_survival(clean_df)}")

+--------------+--------+-------------------+
|smoking_status|patients|survival_rate      |
+--------------+--------+-------------------+
|Never Smoked  |222751  |0.22091034383684025|
|Current Smoker|221898  |0.2203399760250205 |
|Passive Smoker|223170  |0.2200250929784469 |
|Former Smoker |222181  |0.21964074335789288|
+--------------+--------+-------------------+

Highest survival rate smoking group: Never Smoked


## Task 4 — Top 3 countries by percentage of Stage IV diagnoses

In [10]:
def top_countries_stage_iv(df: DataFrame, n: int = 3) -> DataFrame:
    return (
        df.groupBy("country")
        .agg(
            F.count("*").alias("total_patients"),
            F.sum(F.when(F.col("cancer_stage") == "Stage IV", 1).otherwise(0)).alias("stage_iv_patients"),
        )
        .withColumn(
            "stage_iv_pct",
            F.round(F.col("stage_iv_patients") / F.col("total_patients") * 100, 2),
        )
        .orderBy(F.col("stage_iv_pct").desc(), F.col("stage_iv_patients").desc())
        .limit(n)
    )


top_countries_stage_iv(clean_df).show(truncate=False)

+--------------+--------------+-----------------+------------+
|country       |total_patients|stage_iv_patients|stage_iv_pct|
+--------------+--------------+-----------------+------------+
|Greece        |33052         |8429             |25.5        |
|Croatia       |33138         |8426             |25.43       |
|Czech Republic|32885         |8317             |25.29       |
+--------------+--------------+-----------------+------------+



## Task 5 — Filtered cohort: average age and hypertension %


In [11]:
def high_risk_survivor_stats(df: DataFrame) -> dict:
    cohort = df.filter(
        (F.col("gender") == "Male")
        & (F.col("cancer_stage").isin("Stage III", "Stage IV"))
        & (F.col("family_history") == 1)
        & (F.col("smoking_status") == "Current Smoker")
        & (F.col("bmi") > 30)
        & (F.col("survived") == 1)
    )

    stats = cohort.agg(
        F.count("*").alias("cohort_size"),
        F.avg("age").alias("avg_age"),
        F.avg("hypertension").alias("hypertension_rate"),
    ).first()

    return {
        "cohort_size": int(stats["cohort_size"]),
        "average_age": float(stats["avg_age"]) if stats["avg_age"] is not None else None,
        "hypertension_pct": float(stats["hypertension_rate"]) * 100 if stats["hypertension_rate"] is not None else None,
    }


result = high_risk_survivor_stats(clean_df)
print(f"Cohort size:       {result['cohort_size']:,}")
print(f"Average age:       {result['average_age']:.2f}")
print(f"Hypertension %:    {result['hypertension_pct']:.2f}%")

Cohort size:       3,194
Average age:       55.18
Hypertension %:    74.77%


In [12]:
spark.stop()